In [ ]:
!pip install yfinance pandas numpy google-cloud-bigquery

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from google.cloud import bigquery

In [ ]:
data = yf.download(["TSLA", "GME"], start="2018-01-01")

data = data.reset_index()
data.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in data.columns]
data = data.sort_values("Date_")

data.head()

/tmp/ipython-input-364/3524639770.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(["TSLA", "GME"], start="2018-01-01")
[*********************100%***********************]  2 of 2 completed


,Date_,Close_GME,Close_TSLA,High_GME,High_TSLA,Low_GME,Low_TSLA,Open_GME,Open_TSLA,Volume_GME,Volume_TSLA
0,2018-01-02,3.988464,21.368668,3.995017,21.474001,3.883620,20.733334,3.922936,20.799999,11330800,65283000
1,2018-01-03,3.975359,21.150000,4.012491,21.683332,3.914199,21.036667,3.995017,21.400000,15156800,67822500
2,2018-01-04,4.001570,20.974667,4.014675,21.236668,3.922936,20.378668,3.975359,20.858000,11125200,149194500
3,2018-01-05,4.080204,21.105333,4.091125,21.149332,3.979727,20.799999,4.014675,21.108000,12076000,68868000
4,2018-01-08,4.200338,22.427334,4.237470,22.468000,4.106415,21.033333,4.106415,21.066668,14673600,147891000


In [ ]:
data["TSLA_Return"] = data["Close_TSLA"].pct_change()
data["GME_Return"] = data["Close_GME"].pct_change()

In [ ]:
data["TSLA_Log_Return"] = np.log(data["Close_TSLA"] / data["Close_TSLA"].shift(1))
data["GME_Log_Return"] = np.log(data["Close_GME"] / data["Close_GME"].shift(1))

In [ ]:
data["TSLA_MA_50"] = data["Close_TSLA"].rolling(50).mean()
data["GME_MA_50"] = data["Close_GME"].rolling(50).mean()

data["TSLA_MA_200"] = data["Close_TSLA"].rolling(200).mean()
data["GME_MA_200"] = data["Close_GME"].rolling(200).mean()

In [ ]:
data["TSLA_Vol_30"] = data["TSLA_Return"].rolling(30).std()
data["GME_Vol_30"] = data["GME_Return"].rolling(30).std()

data["TSLA_Vol_90"] = data["TSLA_Return"].rolling(90).std()
data["GME_Vol_90"] = data["GME_Return"].rolling(90).std()


In [ ]:
data["TSLA_Cumulative"] = (1 + data["TSLA_Return"]).cumprod()
data["GME_Cumulative"] = (1 + data["GME_Return"]).cumprod()

In [ ]:
data["TSLA_Drawdown"] = data["TSLA_Cumulative"] / data["TSLA_Cumulative"].cummax() - 1
data["GME_Drawdown"] = data["GME_Cumulative"] / data["GME_Cumulative"].cummax() - 1

In [ ]:
data["Portfolio_Return"] = 0.5 * data["TSLA_Return"] + 0.5 * data["GME_Return"]
data["Portfolio_Cumulative"] = (1 + data["Portfolio_Return"]).cumprod()

In [ ]:
project_id = "stock-analytics-pipeline"
client = bigquery.Client(project=project_id)

dataset_id = f"{project_id}.stocks_dataset1"
table_id = f"{dataset_id}.advanced_stock_data"

# Ensure the dataset exists
dataset = bigquery.Dataset(dataset_id)
dataset.location = "asia-south1"
client.create_dataset(dataset, exists_ok=True)

# Upload the data
job = client.load_table_from_dataframe(data, table_id)
job.result()

print("Upload Complete")

Upload Complete
